# EPC 03

In [173]:
import pandas as pd
import random
import math

In [174]:
#method train_test_split

In [175]:
#cross-validation

In [176]:
class PerceptronMultiLayer():
    """
    input(
        layers = [l1, l2, l3, l4]
    )
    """
    #method fit para treinamento
    def __init__(self, layers, learning_rate, weights = None):
        self.layers = layers
        self.learning_rate = learning_rate
        self.weights = weights

        if self.weights == None:
            self.weights = []

            for i in range(len(layers) - 1):
                input_size = layers[i] + 1 #bias
                output_size = layers[i + 1]

                matrix = [
                    [random.uniform(0, 1) for _ in range(input_size)]
                    for _ in range(output_size)
                ]

                self.weights.append(matrix)

    def sigmoid(self,u, beta=0.5):
        return 1 / (1 + math.exp(-beta * u))

    def sigmoid_derivative(self, u, beta=0.5):
        g = self.sigmoid(u, beta)
        return beta * g * (1 - g)
    
    def forward(self, X):
        """
            Recebe os dados de entrada, multiplica pelas entradas,
            soma os biases através do peso associado ao -1
            e calcula camada por camada.
        """
        current = X.copy()

        self.activations = [current]
        self.u_values = []
        #beta 
        for W in self.weights:
            # add o bias em todas as camadas
            current_with_bias = [-1] + current

            next_layer = []
            layer_u = []

            for neuron_weights in W:
                u = 0

                for i in range(len(current_with_bias)):
                    u += current_with_bias[i] * neuron_weights[i]

                layer_u.append(u)
                output = self.sigmoid(u)
                next_layer.append(output)

            #valores do combinador - aI_j/aW_ji
            self.u_values.append(layer_u)

            #resultado do combinador após passar pela função de ativação - aY_j/aI_j
            self.activations.append(next_layer)
            current = next_layer

        return current 

    
    def backward(self, X, y, output):

        # error da camada de saída

        output_delta = []

        for j in range(len(output)):

            delta = (
                output[j] - y[j]
            ) * self.sigmoid_derivative(
                self.u_values[-1][j]
            )

            output_delta.append(delta)


        # erro da camada escondida
        hidden_delta = []

        for i in range(len(self.activations[-2])):

            error = 0

            for j in range(len(output_delta)):

                weight = self.weights[-1][j][i + 1]

                error += weight * output_delta[j]

            delta = (
                error
                * self.sigmoid_derivative(
                    self.u_values[-2][i]
                )
            )

            hidden_delta.append(delta)

        # atualiza hidden → ouput
        previous_activation = self.activations[-2]

        for j in range(len(output_delta)):

            for i in range(len(self.weights[-1][j])):

                if i == 0:
                    activation = -1
                else:
                    activation = previous_activation[i - 1]

                self.weights[-1][j][i] -= (
                    self.learning_rate
                    * output_delta[j]
                    * activation
                )


        # atualiza entradas → escondida

        input_activation = self.activations[0]

        for j in range(len(hidden_delta)):

            for i in range(len(self.weights[0][j])):

                if i == 0:
                    activation = -1
                else:
                    activation = input_activation[i - 1]

                self.weights[0][j][i] -= (
                    self.learning_rate
                    * hidden_delta[j]
                    * activation
                )
                
    def fit(self, X, y, epochs=1):
        """
            Executa o ciclo (forward e backward) repetidamente - parando quando um erro aceitável for encontrado ou alcançar o número máximo de epocas.
            Também atualiza os valores dos pesos e bias a cada iteração para reduzir o erro
        """
        for epoch in range(epochs):
            for i in range(len(X)):
                output = self.forward(X[i])
                self.backward(X[i], y[i], output)

        #depois retornar epoca total, epocas e suas respectivos erros
       
    #method predict para predição
    def predict(self, X):
        return self.forward(X)
    
    #methods auxiliares para calculos de pesos eqm etc

In [177]:
input_data = []
output_data= []

with open('data/iris.dat', 'r', encoding='utf-8') as file:

    for linha in file:
        linha = linha.strip()

        if linha == "@data":
            break

    for linha in file:
        dados = linha.split(",")


        input_data.append([
            float(dados[0]),
            float(dados[1]),
            float(dados[2]),
            float(dados[3]),

        ])
        output_config = [
            [0, 0],
            [0, 1],
            [1, 0]
        ]
        output_data_raw = dados[4].strip()
        match output_data_raw: 
            case 'Iris-setosa':
                output_data.append(output_config[0])
            case 'Iris-versicolor':
                output_data.append(output_config[1])
            case _:
                output_data.append(output_config[2])

print("Inputs:", input_data)
print("Outputs:", output_data)

Inputs: [[5.1, 3.5, 1.4, 0.2], [4.9, 3.0, 1.4, 0.2], [4.6, 3.1, 1.5, 0.2], [5.0, 3.6, 1.4, 0.2], [5.4, 3.9, 1.7, 0.4], [4.6, 3.4, 1.4, 0.3], [5.0, 3.4, 1.5, 0.2], [4.4, 2.9, 1.4, 0.2], [5.4, 3.7, 1.5, 0.2], [4.8, 3.4, 1.6, 0.2], [4.8, 3.0, 1.4, 0.1], [4.3, 3.0, 1.1, 0.1], [5.7, 4.4, 1.5, 0.4], [5.4, 3.9, 1.3, 0.4], [5.1, 3.5, 1.4, 0.3], [5.7, 3.8, 1.7, 0.3], [5.1, 3.8, 1.5, 0.3], [5.4, 3.4, 1.7, 0.2], [5.1, 3.7, 1.5, 0.4], [4.6, 3.6, 1.0, 0.2], [5.1, 3.3, 1.7, 0.5], [4.8, 3.4, 1.9, 0.2], [5.0, 3.0, 1.6, 0.2], [5.0, 3.4, 1.6, 0.4], [5.2, 3.5, 1.5, 0.2], [5.2, 3.4, 1.4, 0.2], [4.7, 3.2, 1.6, 0.2], [4.8, 3.1, 1.6, 0.2], [5.4, 3.4, 1.5, 0.4], [5.2, 4.1, 1.5, 0.1], [5.5, 4.2, 1.4, 0.2], [4.9, 3.1, 1.5, 0.1], [5.0, 3.2, 1.2, 0.2], [5.5, 3.5, 1.3, 0.2], [4.9, 3.1, 1.5, 0.1], [4.4, 3.0, 1.3, 0.2], [5.1, 3.4, 1.5, 0.2], [5.0, 3.5, 1.3, 0.3], [4.5, 2.3, 1.3, 0.3], [5.1, 3.8, 1.9, 0.4], [4.8, 3.0, 1.4, 0.3], [5.1, 3.8, 1.6, 0.2], [4.6, 3.2, 1.4, 0.2], [5.3, 3.7, 1.5, 0.2], [5.0, 3.3, 1.4, 0.2], [

In [ ]:
#to-do: train_test_split
def train_test_split(X, y, test_size=0.2, random_state = 42):

    if len(X) != len(y):
        raise ValueError("X and y must be the same length")

    generator = random.Random(random_state)

    indexs = list(range(len(X)))
    generator.shuffle(indexs)

    tot_items = len(X)
    test_size = int(tot_items * test_size)

    #pega até o tamanho de teste, depois o que sobrou é o treinamento
    indexs_test = indexs[:test_size]
    indexs_train = indexs[test_size:]

    X_train = [X[i] for i in indexs_train]
    X_test = [X[i] for i in indexs_test]
    y_train = [y[i] for i in indexs_train]
    y_test = [y[i] for i in indexs_train]

    return X_train, X_test, y_train, y_test

In [ ]:
mlp = PerceptronMultiLayer([4, 2, 2], 2)

mlp.fit(input_data, output_data, 122)


#to-do: validacao cruzada

#to-do: treinamento com a melhor topologia 
#to-do: conversão com o y